# Nozzle linear systems with Chebyshev LCU

Solve the original eight- and sixteen-row systems with native block encodings and compare the postselected vectors with a direct solve.

## The inverse is an explicit LCU walk with normalization

The displayed Circuit walk composes the matrix encoding, coefficient loader
and controlled odd powers. FlagQuantum executes this explicit composition;
only the approximate coefficient-loader branch uses native synth. Host code
derives reciprocal coefficients, alternating signs, spectral cutoff and
normalizations for the small and large nozzle systems.

Inspect all three solves: small degree 31, large degree 255 and the large
approximate-loader variant. Solution rescaling and selector-zero success
probability matter as much as normalized shape. The independent full-state
reference is separate from the direct linear solve. A loader tolerance does
not equal solution error. This remains a mixed Circuit/synth workflow, not
a high-level compiler deriving an inverse algorithm from a matrix.


Use the local FlagQuantum environment.

In [ ]:
import flagquantum as fq
import torch

from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_chebyshev_lcu.overview.workflow import (
    CTYPE,
    DTYPE,
    Encoding,
    preparation,
)


def walk_preview(
    encoding: Encoding,
    b: torch.Tensor,
    rhs: torch.Tensor,
    prep: fq.Circuit,
    inverse_width: int,
) -> fq.Circuit:
    width = encoding.data_width + encoding.block_width
    size = 1 << width
    block_zero = 1 << encoding.data_width
    reflection_signs = -torch.ones(size, dtype=DTYPE)
    reflection_signs[:block_zero] = 1.0
    reflection = torch.diag(reflection_signs).to(CTYPE)
    symmetrized = b @ reflection @ b.mH @ reflection
    selector = tuple(range(inverse_width))
    encoded = tuple(range(inverse_width, inverse_width + width))
    data = encoded[-encoding.data_width :]
    circuit = fq.Circuit(inverse_width + width, dtype=CTYPE)
    circuit.compose(prep, qubits=selector)
    circuit.compose(preparation(rhs / rhs.norm()), qubits=data)
    circuit.z(selector[-1])
    controlled = torch.eye(2 * size, dtype=CTYPE)
    for wire in reversed(selector):
        # One exact powered block for each original binary control.
        controlled[size:, size:] = symmetrized
        circuit.any(wire, *encoded, matrix=controlled.clone())
        symmetrized = symmetrized @ symmetrized
    circuit.any(*encoded, matrix=reflection @ b.mH @ reflection)
    circuit.compose(prep.adjoint(), qubits=selector)
    return circuit


from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq


Import the native inverse solve workflow.

In [ ]:
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview.model import (
    source_matrix,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_chebyshev_lcu.overview.main import (
    plot_large,
    plot_small,
    print_report,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_chebyshev_lcu.overview.workflow import (
    input_vector,
    solve,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.model import (
    source_matrix as small_matrix,
)

print(fq.__name__)


Apply one signed odd Chebyshev power for each selector index.

In [ ]:
print("Small and large maximum odd powers:", 31, 255)


Postselect both LCU registers at zero and recover the signed solution.

In [ ]:
print("Measurement condition: inverse selector = matrix selector = 0")


Set the output directory.

In [ ]:
figures = Path("notebook-figures")


Read the original eight-row nozzle system.

In [ ]:
a_small, b_small = small_matrix(), input_vector("b_nozzle_small", 3)
print(a_small.shape, b_small.shape)


Keep the four-bit small selector.

In [ ]:
print("Small LCU selector width: 4")


Run the degree-31 Pauli encoding with all 16 inverse coefficients.

In [ ]:
small = solve(False)
print_report("Small", small)


Compare the computed vector with the dense solution.

In [ ]:
plot_small(small, figures)
display(Image(filename=str(figures / "nozzle-small.png")))


Check the source small-system error bound.

In [ ]:
assert small.solution_error < 0.2


Read the original sixteen-row nozzle system.

In [ ]:
a_large, b_large = source_matrix(), input_vector("nozzle_008_b", 4)
print(a_large.shape, b_large.shape)


Use the full seven-bit selector for both large preparations.

In [ ]:
print("Large LCU selector width: 7")


Execute the exact coefficient preparation and degree-255 banded solve.

In [ ]:
exact = solve(True)
print_report("Large exact", exact)


Execute the factorized coefficient preparation with the source error budget.

In [ ]:
approximate = solve(True, 0.03)
print_report("Large approximate", approximate)


Plot both estimates and the dense reference.

In [ ]:
plot_large(exact, approximate, figures)
display(Image(filename=str(figures / "nozzle-008.png")))


Check both source large-system error limits.

In [ ]:
assert exact.solution_error < 0.1
assert approximate.solution_error < 0.1
